# Chopin the Ⅲ ― 楽譜読み取りAIの学習（Google Colab 用）

このノートブックは、上から順に ▶（実行）を押していくだけで、楽譜読み取りAIの学習ができます。

**最初に1回だけ：GPU を選ぶ**
メニューの「ランタイム」→「ランタイムのタイプを変更」→「ハードウェア アクセラレータ」で **A100 GPU**（無ければ **L4 GPU**）を選んで「保存」。

**途中で止まっても大丈夫**
学習の途中経過は Google ドライブの「chopin-omr」フォルダに保存されます。止まったら、もう一度上から実行すると続きから再開します。

## 1. Google ドライブにつなぐ（保存先）
実行すると、Google アカウントの許可を求める画面が出ます。「許可」を押してください。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE = '/content/drive/MyDrive/chopin-omr'   # 学習の結果を置く場所（変えなくてよい）
RUN = SAVE + '/run1'                          # 学習の名前（作り直すときは run2 などに変える）
os.makedirs(RUN, exist_ok=True)
print('保存先:', RUN)

## 2. GPU の確認
下に `A100` や `L4` と出れば準備できています。`NVIDIA-SMI has failed` と出たら、上の「最初に1回だけ」をやり直してください。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. プログラムを取ってくる・必要なものを入れる（1〜2分）

In [ ]:
!rm -rf /content/pkm && git clone -q --depth 1 https://github.com/hirosak514/pianokeymarker /content/pkm
%cd /content/pkm/omr-train
!pip install -q verovio cairosvg onnx onnxruntime
import torch; print('PyTorch', torch.__version__, '/ GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'なし')

## 4. 練習用の楽譜画像を作る
乱数で作った曲（著作権の問題が無い曲）を楽譜画像にし、写真のようなゆがみ・影・ぼけを加えます。
- `COUNT` が練習問題の数です。最初は **200000**（20万枚）がおすすめです。A100 の環境で 30〜60分ほどかかります。
- 作った画像は Google ドライブにも保存するので（約3GB）、2回目からは作り直さずにすぐ使えます。

In [ ]:
COUNT = 200000
import os, subprocess
DATA = '/content/data'
ZIP = SAVE + f'/data_{COUNT}.zip'
if os.path.exists(DATA + '/train/labels.tsv'):
    print('作ってある画像を使います')
elif os.path.exists(ZIP):
    print('ドライブに保存してある画像を取り出します…')
    !mkdir -p /content/data && cd /content/data && unzip -q "$ZIP"
else:
    n = os.cpu_count()
    print(f'CPU {n}個で作ります')
    !python -u -m omr.make_data --out /content/data/val --count 2000 --seed 999 --workers {n}
    !python -u -m omr.make_data --out /content/data/train --count {COUNT} --workers {n}
    print('ドライブに保存しています…')
    !cd /content/data && zip -q -r -0 "$ZIP" train val
!wc -l /content/data/train/labels.tsv /content/data/val/labels.tsv

## 5. 学習する（いちばん時間がかかります）
- `HOURS` の時間だけ学習して止まります（Colab の1回の利用時間に合わせて、9時間にしています）。
- 2000歩ごとに「検証: 記号の間違い ○%」と出ます。この数字が小さくなっていけば、うまく覚えています。
- 止まったら（時間切れ・切断など）、もう一度 **1 から順に** 実行すれば、続きから再開します。

In [ ]:
HOURS = 9
import os
!python -u -m omr.train --data /content/data/train --val /content/data/val --out "$RUN" --hours {HOURS} --workers {os.cpu_count()}

## 6. 学習の進み具合をグラフで見る
左のグラフ（記号の間違いの割合）が下がり、右のグラフ（1段を完全に正しく読めた割合）が上がっていれば順調です。

In [ ]:
import json, matplotlib.pyplot as plt
rows = [json.loads(l) for l in open(RUN + '/log.jsonl')]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot([r['step'] for r in rows], [r['ser'] * 100 for r in rows]); ax[0].set_title('symbol error % (lower is better)'); ax[0].set_xlabel('step')
ax[1].plot([r['step'] for r in rows], [r['exact'] * 100 for r in rows]); ax[1].set_title('systems read perfectly %'); ax[1].set_xlabel('step')
plt.show()
print('最新:', rows[-1])

## 7. アプリで使う形（ONNX）に書き出す
ドライブの `chopin-omr/run1/onnx` フォルダに `encoder.onnx`・`decoder.onnx`・`vocab.json` ができます。
**この3つのファイル**をチャットに添付してください（アプリに組み込みます）。

In [ ]:
!python -m omr.export --ckpt "$RUN/best.pt" --out "$RUN/onnx"
!ls -la "$RUN/onnx"

## 8. 自分の楽譜で試す（任意）
大譜表**1段分**を切り出した画像（スクリーンショットでも可）を選ぶと、読み取った記号を表示します。

In [ ]:
from google.colab import files
up = files.upload()
for name in up:
    !python -m omr.predict --ckpt "$RUN/best.pt" "$name"